# GPU Inference Residency Plan

This notebook prepares the GPU inference experiment for NeuralZip compressed residency. It has an explicit CUDA guard and `RUN_GPU_EXPERIMENT = False` by default, so opening or validating the notebook will not load a large model or launch GPU work.

In [1]:
from pathlib import Path
import sys

NOTEBOOK_DIR = Path.cwd().resolve()
if (NOTEBOOK_DIR / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR
elif (NOTEBOOK_DIR.parent / "experiment_utils.py").exists():
    REPO_ROOT = NOTEBOOK_DIR.parent
else:
    REPO_ROOT = Path("..").resolve()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

EXAMPLES_DIR = REPO_ROOT / "examples"
OUTPUT_DIR = EXAMPLES_DIR / "outputs"

import json

import pandas as pd
import torch

from model_compression import HuggingFaceModel

try:
    from IPython.display import display
except ImportError:
    display = print

/Users/martin/miniconda3/envs/dev/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
RUN_GPU_EXPERIMENT = False
GPU_MODEL_ID = "meta-llama/Llama-3.2-1B"
GPU_DTYPE = "bfloat16"
PLAN_KWARGS = {"ngram": 3, "block_size": 4096, "threshold": 0.02}

if RUN_GPU_EXPERIMENT:
    if not torch.cuda.is_available():
        raise RuntimeError("CUDA is not available; leave this notebook prepared without running the GPU experiment.")

    model = HuggingFaceModel(GPU_MODEL_ID, dtype=GPU_DTYPE)
    model.model.to("cuda")
    plan = model.build_gpu_inference_plan(**PLAN_KWARGS)
    plan_summary = pd.DataFrame([
        {
            "model_id": GPU_MODEL_ID,
            "dtype": GPU_DTYPE,
            "ngram": plan["ngram"],
            "block_size": plan["block_size"],
            "tensors": len(plan["tensors"]),
            "clusters": len(plan["clusters"]),
            "estimated_total_bytes": plan["estimated_total_bytes"],
            "metadata_bytes": plan["metadata_bytes"],
            "tail_bytes": plan["tail_bytes"],
            "decode_policy": plan["decode_policy"],
        }
    ])
    display(plan_summary)
else:
    display(pd.DataFrame([
        {
            "status": "prepared_not_executed",
            "cuda_available": bool(torch.cuda.is_available()),
            "model_id": GPU_MODEL_ID,
            "dtype": GPU_DTYPE,
            "guard": "set RUN_GPU_EXPERIMENT=True only on a CUDA machine with model access",
        }
    ]))

,status,cuda_available,model_id,dtype,guard
0,prepared_not_executed,False,meta-llama/Llama-3.2-1B,bfloat16,set RUN_GPU_EXPERIMENT=True only on a CUDA mac...


In [3]:
# Optional plan export after a real GPU run.
EXPORT_GPU_PLAN = False
if RUN_GPU_EXPERIMENT and EXPORT_GPU_PLAN:
    plan_path = OUTPUT_DIR / "gpu_inference_plan.json"
    plan_path.write_text(json.dumps(plan, indent=2))
    plan_path